# V16 – Lösungen zu 03_aufgaben

Vollständige Lösungen zu den fünf Aufgaben aus [../03_aufgaben.ipynb](../03_aufgaben.ipynb). Alle Pfade sind relativ zum `loesungen/`-Ordner (`../` geht eine Ebene höher in den V16-Ordner).

## A1 – Messwerte-Anzahl

`df.shape[0]` liefert die Anzahl der Datenzeilen ohne Kopfzeile.

In [1]:
import pandas as pd

df = pd.read_csv("../sensoren_daten.csv")
anzahl_messungen = df.shape[0]
print(f"anzahl_messungen = {anzahl_messungen}")

try:
    assert anzahl_messungen == 15
    print("✅ A1 richtig.")
except AssertionError as e:
    print(f"❌ {e}")

anzahl_messungen = 15
✅ A1 richtig.


## A2 – Mittelwert pro Spalte

Einzeiler: `df[spalte].mean()` als Rückgabewert.

In [2]:
def spalten_mittel(df, spalte):
    """Liefert den Mittelwert der angegebenen Spalte."""
    return float(df[spalte].mean())

ergebnis = spalten_mittel(df, "Temperatur_C")
print(f"spalten_mittel(df, 'Temperatur_C') = {ergebnis:.2f}")

try:
    assert abs(ergebnis - 53.7) < 0.01
    assert abs(spalten_mittel(df, "Druck_bar") - 50.726) < 0.1
    print("✅ A2 richtig.")
except AssertionError as e:
    print(f"❌ {e}")

spalten_mittel(df, 'Temperatur_C') = 53.70
✅ A2 richtig.


## A3 – Offene Wartungsaufträge

JSON hat eine Root-Ebene `"wartungsauftraege"`; wir lesen mit `json.load` und bauen das DataFrame aus der Liste. Anschließend Boolean-Maske auf `status == "Geplant"` und Länge messen.

In [3]:
import json

with open("../wartungsauftraege.json", encoding="utf-8") as f:
    daten = json.load(f)

df_w = pd.DataFrame(daten["wartungsauftraege"])
offene_auftraege = len(df_w[df_w["status"] == "Geplant"])
print(f"offene_auftraege = {offene_auftraege}")

try:
    assert offene_auftraege == 6
    print("✅ A3 richtig.")
except AssertionError as e:
    print(f"❌ {e}")

offene_auftraege = 6
✅ A3 richtig.


## A4 – Produktionslinie mit höchster Stückzahl

Für jede `produktionslinie` iterieren wir durch ihre `auftrag`-Kinder und sammeln `linie_id` und `stueckzahl_produziert` in einer Liste von Dictionaries. Danach in ein DataFrame wandeln, gruppieren, summieren, `idxmax`.

In [4]:
import xml.etree.ElementTree as ET

baum = ET.parse("../produktionsplanung.xml")

zeilen = []
for linie in baum.findall(".//produktionslinie"):
    linie_id = linie.get("id")
    for auftrag in linie.findall("auftrag"):
        zeilen.append({
            "linie_id": linie_id,
            "auftrag_id": auftrag.get("auftrag_id"),
            "stueckzahl_produziert": int(auftrag.findtext("stueckzahl_produziert")),
        })

df_prod = pd.DataFrame(zeilen)
summen = df_prod.groupby("linie_id")["stueckzahl_produziert"].sum()
print(summen)

beste_linie = summen.idxmax()
print(f"beste_linie = {beste_linie}")

try:
    assert beste_linie == "Linie_B"
    print("✅ A4 richtig.")
except AssertionError as e:
    print(f"❌ {e}")

linie_id
Linie_A    433
Linie_B    682
Linie_C     82
Name: stueckzahl_produziert, dtype: int64
beste_linie = Linie_B
✅ A4 richtig.


## A5 – Teuerste Produkt-Kategorie

Standard-`groupby`-Muster: nach Kategorie gruppieren, Mittel des Preises, Index des Maximums.

In [5]:
df_kat = pd.read_csv("../produkte_katalog.csv")
mittelpreis = df_kat.groupby("Kategorie")["Preis_Euro"].mean().sort_values(ascending=False)
print(mittelpreis)

teuerste_kategorie = mittelpreis.idxmax()
print(f"teuerste_kategorie = {teuerste_kategorie}")

try:
    assert teuerste_kategorie == "Hydraulik"
    print("✅ A5 richtig.")
except AssertionError as e:
    print(f"❌ {e}")

Kategorie
Hydraulik              285.000000
Führungssysteme        125.500000
Pneumatik               68.500000
Antriebstechnik         50.233333
Zerspanungswerkzeug     32.800000
Werkzeugaufnahme        18.750000
Lager                   12.400000
Verbindungselemente      2.350000
Name: Preis_Euro, dtype: float64
teuerste_kategorie = Hydraulik
✅ A5 richtig.
